# Prepare data
This notebook serve the purpose of preparing the data for the training of the model. It includes dataset formatting to the right format, splitting the dataset, cleaning it by removing the cases that we marked as too damaged, computing the common foreground mask, and statistics about the training data that will be used for the data augmentation and the training of the model.

## DRIVE dataset

The process is kinda the same for the DRIVE dataset, which comes in a different format so we need to format it to the same architecture we used for FIVES

In [ ]:
import os

drive = AvailableDatasets.DRIVE.value

drive_dataset_dir = drive.get_data_dir()
print(f"DRIVE dataset directory: {drive_dataset_dir}")

In [ ]:
import json
from tqdm import tqdm
from PIL import Image
import torch
import numpy as np

def process_split(main_dir, split_name):
    # --- Define source and destination directories ---
    src_img_dir = os.path.join(main_dir, split_name, "images")
    src_gt_dir = os.path.join(main_dir, split_name, "1st_manual")
    src_foreground_mask_dir = os.path.join(main_dir, split_name, "mask")

    dst_img_dir = os.path.join(main_dir, "img")
    dst_gt_dir = os.path.join(main_dir, "gt")
    dst_foreground_mask_dir = os.path.join(main_dir, "foreground_masks")
    os.makedirs(dst_img_dir, exist_ok=True)
    os.makedirs(dst_gt_dir, exist_ok=True)
    os.makedirs(dst_foreground_mask_dir, exist_ok=True)

    # --- List image, ground truth, and foreground mask filenames ---
    img_filenames_list = [f.split('.')[0] for f in os.listdir(src_img_dir) if f.endswith('.tif')]

    # --- Process each image, ground truth, and foreground mask ---
    idx_list = []
    for img_filename in tqdm(img_filenames_list):
        i = int(img_filename.split('_')[0])
        gt_filename = f"{i:02d}_manual1"
        foreground_mask_filename = f"{i:02d}_{split_name}_mask"

        id_name = f"DRIVE_{i:03d}"
        idx_list.append(id_name)

        # --- Convert image (.tif -> .png) ---
        src_img_path = os.path.join(src_img_dir, f"{img_filename}.tif")
        dst_img_path = os.path.join(dst_img_dir, f"{id_name}.png")
        img = Image.open(src_img_path)
        img.save(dst_img_path)

        # --- Convert ground truth (.gif -> .png) ---
        src_gt_path = os.path.join(src_gt_dir, f"{gt_filename}.gif")
        dst_gt_path = os.path.join(dst_gt_dir, f"{id_name}.png")
        gt = Image.open(src_gt_path)
        gt.save(dst_gt_path)

        # --- Convert foreground mask (.gif -> .pt) ---
        src_foreground_mask_path = os.path.join(src_foreground_mask_dir, f"{foreground_mask_filename}.gif")
        dst_foreground_mask_path = os.path.join(dst_foreground_mask_dir, f"{id_name}.pt")
        foreground_mask = Image.open(src_foreground_mask_path)
        foreground_mask = torch.tensor(np.array(foreground_mask) > 0, dtype=torch.uint8)
        torch.save(foreground_mask, dst_foreground_mask_path)

    return idx_list


# === Process splits ===
drive_splits = {}
drive_train_idx = process_split(drive_dataset_dir, "training")
drive_test_idx = process_split(drive_dataset_dir, "test")
drive_splits['train'] = drive_train_idx
drive_splits['test'] = drive_test_idx

# === Save splits ===
drive_splits_filepath = os.path.join(drive_dataset_dir, "splits.json")
with open(drive_splits_filepath, 'w') as f:
    json.dump(drive_splits, f, indent=4)

In [ ]:
from image_segmentation.data import ImageDataset, ImageDatamodule

drive_dataset = ImageDataset(data_dir=drive_dataset_dir, transforms=None)
drive_datamodule = ImageDatamodule(drive_dataset, 
                             split_file_path=drive_splits_filepath,
                             train_split_name='train',
                             val_split_ratio=0.2,
                             train_transforms=None,
                             val_transforms=None,
                             test_transforms=None,
                             num_workers=0,
                             train_batch_size=4,
                             val_batch_size=1,
                             seed=42,
                             shuffle_train=True)
drive_datamodule.setup()

In [ ]:
# We use a FOV of 45 degrees for the DRIVE dataset, as specified in the dataset documentation.
stats = drive_dataset.get_dataset_stats(split_name='train', split_indices=drive_datamodule.train_indices.tolist() + drive_datamodule.val_indices.tolist(), fov = 45)
print("Dataset Statistics for 'train' split:")
print(stats)

In [ ]:
import matplotlib.pyplot as plt

drive_train_dataloader = drive_datamodule.train_dataloader()
img, gt = next(iter(drive_train_dataloader))
print(f"Image batch shape: {img.shape}")
print(f"Ground truth batch shape: {gt.shape}")

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img[0].numpy())
plt.title("Sample Image")
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(gt[0].squeeze().numpy(), cmap='gray')
plt.title("Sample Ground Truth")
plt.axis('off')
plt.show()

Now that the datasets are in the right format, you can continue on the [U-Net pretraining notebook (2)](./02_pretrain_unet.ipynb)